# T2.4 - View Definitions
Owner: D (El Dib Yehea)
Creates the ML pipeline views in DBRepo via the REST API.

In [30]:
import requests
import os
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

BASE_URL    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = os.getenv("DBREPO_USERNAME")
PASSWORD    = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = "82c19b39-246c-4409-b25c-8baf3a158a70"
TABLE_ID    = "9e7a7b18-58d9-4053-864e-82232463c8f5"


In [31]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    db = response.json()
    print(f"Connection successful")
    print(f"Database name : {db.get('name')}")
    print(f"Database ID   : {db.get('id')}")
else:
    print(f"Connection failed with status {response.status_code}")
    print(response.text[:200])

Connection successful
Database name : uk-collision-severity-prediction-main
Database ID   : 82c19b39-246c-4409-b25c-8baf3a158a70


In [32]:
VIEWS = [
    {
        "name": "collision_ml_features",
        "purpose": "Clean feature table matching exactly the input expected by 01_load_data.py. Contains the 15 features and label used for ML training. Primary source for T2.6 API reimplementation.",
        "body": {
            "name": "collision_ml_features",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "53589346-d32f-4590-b2a6-1d2c29ded1f0"},  # speed_limit
                    {"id": "f484011a-7e4f-4f48-b60e-962aebbfbbc7"},  # light_conditions
                    {"id": "ea88d451-4b33-42ba-84fa-c0c845811562"},  # weather_conditions
                    {"id": "83f66deb-c9e0-4a57-9f17-5862ec6b41a9"},  # road_surface_conditions
                    {"id": "c4053f9d-07d8-470f-9b4a-807f0f363ff4"},  # road_type
                    {"id": "19cab0f5-7cb2-49f8-8636-56fa121a2ba1"},  # urban_or_rural_area
                    {"id": "2b0d2f3f-7f9d-45d4-8f12-4859f40df85a"},  # number_of_vehicles
                    {"id": "25005c3c-28a8-4a51-a989-79abf503c0d1"},  # number_of_casualties
                    {"id": "f3a0998a-28b1-4551-93dc-dd756ea127f1"},  # day_of_week
                    {"id": "82c2b036-4144-475e-afb8-8364ce591f39"},  # junction_detail
                    {"id": "3cccdcda-0ca1-41b1-b681-e7296b3adf24"},  # junction_control
                    {"id": "0d87b5af-c6d7-4046-9c14-2b57708f1189"},  # pedestrian_crossing
                    {"id": "afa7f77b-a3ba-4963-9506-9a6d01ec7a2d"},  # first_road_class
                    {"id": "87206859-6de5-4350-8a14-aa1ba9b28c04"},  # special_conditions_at_site
                    {"id": "2c479d3f-6aa5-40e7-ac94-0fc8393dbfa8"},  # carriageway_hazards
                    {"id": "bca7b890-ffa4-45dd-9de3-3830599ef0eb"},  # collision_severity
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    },
    {
        "name": "collision_severity_summary",
        "purpose": "Aggregated collision counts grouped by severity, road type, urban/rural area and speed limit. Used to verify class imbalance before SMOTE balancing in 02_preprocess.py.",
        "body": {
            "name": "collision_severity_summary",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "bca7b890-ffa4-45dd-9de3-3830599ef0eb"},  # collision_severity
                    {"id": "19cab0f5-7cb2-49f8-8636-56fa121a2ba1"},  # urban_or_rural_area
                    {"id": "c4053f9d-07d8-470f-9b4a-807f0f363ff4"},  # road_type
                    {"id": "53589346-d32f-4590-b2a6-1d2c29ded1f0"},  # speed_limit
                    {"id": "25005c3c-28a8-4a51-a989-79abf503c0d1"},  # number_of_casualties
                    {"id": "2b0d2f3f-7f9d-45d4-8f12-4859f40df85a"},  # number_of_vehicles
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    }
]

print(f"Defined {len(VIEWS)} views and ready to create")
for v in VIEWS:
    print(f"  - {v['name']}: {v['purpose'][:60]}...")

Defined 2 views and ready to create
  - collision_ml_features: Clean feature table matching exactly the input expected by 0...
  - collision_severity_summary: Aggregated collision counts grouped by severity, road type, ...


In [33]:
created_views = {}

for view in VIEWS:
    print(f"Creating view: {view['name']}")

    response = requests.post(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json"
        },
        json=view["body"]
    )

    print(f"  Status: {response.status_code}")

    if response.status_code in (200, 201):
        view_id = response.json().get("id")
        created_views[view["name"]] = view_id
        print(f"  Created successfully with ID: {view_id}")
    else:
        print(f"  Failed: {response.text[:300]}")

print(f"\nTotal views created: {len(created_views)} out of {len(VIEWS)}")

Creating view: collision_ml_features
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}
Creating view: collision_severity_summary
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}

Total views created: 0 out of 2


In [29]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    views_in_db = response.json()
    print(f"Views found in database: {len(views_in_db)}")
    for v in views_in_db:
        print(f"  ID: {v.get('id')} | Name: {v.get('name')}")
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Status: 200
Found 44 columns:
  ID: 8685f21a-0694-4071-95ec-30d5730b706e | Name: collision_index
  ID: f79ea168-d413-4506-951e-50db1f43ee47 | Name: collision_year
  ID: 84758768-d48e-47d4-b5cd-d537e7fe1152 | Name: collision_ref_no
  ID: 6b6d45bc-303a-477b-b748-d4eca8ddaf6b | Name: location_easting_osgr
  ID: b42b0db6-3634-4abe-a648-14f60ce81490 | Name: location_northing_osgr
  ID: 9fe57aef-8051-4b2a-b12b-bdad14b2fcfa | Name: longitude
  ID: 1840b99f-440b-4861-ae8f-ed5f16dca6b2 | Name: latitude
  ID: 6d58a1ba-b2a2-4307-ae9f-4e33a005809a | Name: police_force
  ID: bca7b890-ffa4-45dd-9de3-3830599ef0eb | Name: collision_severity
  ID: 2b0d2f3f-7f9d-45d4-8f12-4859f40df85a | Name: number_of_vehicles
  ID: 25005c3c-28a8-4a51-a989-79abf503c0d1 | Name: number_of_casualties
  ID: d5a6dffe-5cb5-4fc8-8ca5-818101448329 | Name: date
  ID: f3a0998a-28b1-4551-93dc-dd756ea127f1 | Name: day_of_week
  ID: f5843741-2add-4361-920f-4298f1dac5a5 | Name: time
  ID: c7e6d89c-089d-48aa-b517-1df1fbdbfede | Name:

In [ ]:
if created_views.get("collision_ml_features"):
    view_id = created_views["collision_ml_features"]

    response = requests.get(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view/{view_id}/data",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={"Accept": "application/json"},
        params={"page": 0, "size": 5}
    )

    print(f"collision_ml_features spot check - Status: {response.status_code}")

    if response.status_code == 200:
        import pandas as pd
        data = response.json()
        df = pd.DataFrame(data)
        print(f"Rows returned: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        print(df.head())
    else:
        print(f"Failed: {response.text[:200]}")
else:
    print("collision_ml_features was not created, skipping spot check")